    In this notebook, the `transformers` Python package is used to efficiently train a custom model. It covers the following techniques:
    1. Load Model, Tokenizer and Template for Chat Model.
    2. Process Data for Training.
    3. Train Model with LoRA (parameter-efficient fine-tuning).
    4. Evaluate Model's performance.
    5. Save and Deploy Trained Model.

# Installation

In [ ]:
!pip install -q h5py typing-extensions wheel

In [ ]:
!pip uninstall -y unsloth unsloth-zoo transformers huggingface-hub datasets peft diffusers sentence-transformers accelerate


Found existing installation: unsloth 2025.11.3
Uninstalling unsloth-2025.11.3:
  Successfully uninstalled unsloth-2025.11.3
Found existing installation: unsloth_zoo 2025.11.4
Uninstalling unsloth_zoo-2025.11.4:
  Successfully uninstalled unsloth_zoo-2025.11.4
Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
Found existing installation: datasets 4.3.0
Uninstalling datasets-4.3.0:
  Successfully uninstalled datasets-4.3.0
Found existing installation: peft 0.18.0
Uninstalling peft-0.18.0:
  Successfully uninstalled peft-0.18.0
Found existing installation: diffusers 0.35.2
Uninstalling diffusers-0.35.2:
  Successfully uninstalled diffusers-0.35.2
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0


In [ ]:
!pip install "unsloth[zoo]"




  Using cached unsloth-2025.11.3-py3-none-any.whl.metadata (61 kB)
  Using cached unsloth_zoo-2025.11.4-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached huggingface_hub-1.1.5-py3-none-any.whl.metadata (13 kB)
  Using cached diffusers-0.35.2-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached peft-0.18.0-py3-none-any.whl (556 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached unsloth_zoo-2025.11.4-py3-none-any.whl (283 kB)
Using cached diffusers-0.35.2-py3-none-any.whl 

In [ ]:
!pip install unsloth
# Get latest Unsloth
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7ha1yf8k/unsloth_fff9e16fda14401987f20300dbc9aaa9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7ha1yf8k/unsloth_fff9e16fda14401987f20300dbc9aaa9
  Resolved https://github.com/unslothai/unsloth.git to commit 872e191494104d177d854a2e2b430a03a193d4e2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Set Up Hugging Face token

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Get the token from Colab's secret manager
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
login(token=hf_token)


# Load Pre-trained model and tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2-7B", # Reminder we support ANY Hugging Face model!
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.11.3: Fast Qwen2 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# Load Dataset

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("gbharti/finance-alpaca")

full_ds = raw_dataset["train"]

max_train_samples = 10000
full_ds = full_ds.select(range(min(max_train_samples, len(full_ds))))

print("Total samples used:", len(full_ds))
print(full_ds[0])


Total samples used: 10000
{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or

# Data Preparation

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Load the dataset
dataset = load_dataset("gbharti/finance-alpaca", split="train")

# 2. Define Prompt & Formatting Function
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# 3. Apply Mapping ONCE (Cleanest way: Map first, then split)
dataset = dataset.map(formatting_prompts_func, batched = True)

dataset_split = dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = dataset_split["train"]
eval_dataset  = dataset_split["test"]

Map:   0%|          | 0/68912 [00:00<?, ? examples/s]

# Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/65466 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3446 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 65,466 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,2.006500
2,1.979600
3,2.185600
4,1.480900
5,1.476000
6,1.627200
7,1.335500
8,1.349400
9,1.177300
10,1.741100


# Trained Model Results


In [ ]:
import math
final_stats = trainer.evaluate()
final_loss = final_stats["eval_loss"]
final_perplexity = math.exp(final_loss)


print(f"1. final loss: {final_loss:.4f}")
print(f"2. Final Perplexity:    {final_perplexity:.4f}")

1. final loss: 1.2025
2. Final Perplexity:    3.3283


In [ ]:
# 1. Prepare model for inference (makes generation 2x faster)
FastLanguageModel.for_inference(model)

test_questions = [
    "Explain in simple terms what an ETF is and how it differs from a mutual fund.",
    "I have $10,000 and a long-term horizon. How should I think about diversifying my investments?",
    "What is the difference between a fixed-rate mortgage and an adjustable-rate mortgage?",
]

print("=== INFERENCE ===")
for question in test_questions:
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                question, # Instruction
                "",       # Input (empty for these questions)
                ""        # Output (leave empty for generation)
            )
        ],
        return_tensors = "pt"
    ).to("cuda")

    # Generate the answer
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        use_cache = True,
        temperature = 0,
        top_p = 0.9
    )

    # Decode and print
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Clean up the output to just show the response (remove the prompt)
    # We split by "### Response:" to get just the answer
    final_answer = response.split("### Response:")[-1].strip()

    print(f"\nQUESTION: {question}")
    print(f"MODEL ANSWER: {final_answer}")
    print("-" * 60)

=== INFERENCE ===

QUESTION: Explain in simple terms what an ETF is and how it differs from a mutual fund.
MODEL ANSWER: An ETF (Exchange Traded Fund) is a type of investment fund that is traded on a stock exchange. It is similar to a mutual fund in that it holds a portfolio of stocks, bonds, or other securities. However, ETFs are traded on the stock exchange like individual stocks, whereas mutual funds are bought and sold by the fund manager. ETFs are generally more cost-effective and offer more liquidity than mutual funds.
------------------------------------------------------------

QUESTION: I have $10,000 and a long-term horizon. How should I think about diversifying my investments?
MODEL ANSWER: I would recommend investing in a mix of stocks and bonds. You can invest in a mix of stocks and bonds to diversify your portfolio. You can also invest in a mix of different types of stocks and bonds to further diversify your portfolio.
-----------------------------------------------------

In [ ]:
# Save the LoRA adapters
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

print("✅ Model adapters saved to 'lora_model' folder.")

✅ Model adapters saved to 'lora_model' folder.


In [ ]:
import torch
import gc

# 1. Force Clean the GPU Memory
torch.cuda.empty_cache()
gc.collect()

967

# Baseline Model Results

In [ ]:
import math
print(""" BASELINE Perplexity """)
# 2. Use this Context Manager to turn OFF your fine-tuning temporarily
with model.disable_adapter():
    # The trainer will now see the "dumb" base model
    baseline_stats = trainer.evaluate()

baseline_loss = baseline_stats["eval_loss"]
baseline_perplexity = math.exp(baseline_loss)
print(f"1. baseline_loss loss: {baseline_loss:.4f}")
print(f"2. baseline_perplexity:    {baseline_perplexity:.4f}")


 BASELINE Perplexity 


1. baseline_loss loss: 1.9423
2. baseline_perplexity:    6.9748


In [ ]:
# 1. Prepare model for inference (makes generation 2x faster)
FastLanguageModel.for_inference(model)

# 2. Define the questions you want to ask (same as your friend's list)
test_questions = [
    "Explain in simple terms what an ETF is and how it differs from a mutual fund.",
    "I have $10,000 and a long-term horizon. How should I think about diversifying my investments?",
    "What is the difference between a fixed-rate mortgage and an adjustable-rate mortgage?",
]

# 3. The Generation Loop
print("=== INFERENCE ===")

for question in test_questions:
    # Format the input exactly like the training data
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                question, # Instruction
                "",       # Input (empty for these questions)
                ""        # Output (leave empty for generation)
            )
        ],
        return_tensors = "pt"
    ).to("cuda")

    # Generate the answer
    with model.disable_adapter():
      outputs = model.generate(
          **inputs,
          max_new_tokens = 256,
          use_cache = True,
          temperature = 0,
          top_p = 0.9
      )

    # Decode and print
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Clean up the output to just show the response (remove the prompt)
    # We split by "### Response:" to get just the answer
    final_answer = response.split("### Response:")[-1].strip()

    print(f"\nQUESTION: {question}")
    print(f"MODEL ANSWER: {final_answer}")
    print("-" * 60)

=== INFERENCE ===

QUESTION: Explain in simple terms what an ETF is and how it differs from a mutual fund.
MODEL ANSWER: An ETF, or exchange-traded fund, is a type of investment that tracks a specific index, such as the S&P 500 or the Dow Jones Industrial Average. It is similar to a mutual fund in that it pools money from many investors to buy a diversified portfolio of stocks or other assets. However, unlike a mutual fund, an ETF is traded on a stock exchange like a stock, meaning that it can be bought and sold throughout the day at market prices. This makes it more flexible and potentially more liquid than a mutual fund, but it also means that it may be subject to more volatility.
------------------------------------------------------------

QUESTION: I have $10,000 and a long-term horizon. How should I think about diversifying my investments?
MODEL ANSWER: To diversify your investments with $10,000 and a long-term horizon, you should consider a mix of different asset classes such as